# Notebook 2: R-GCN Model Training — Personalized Drug Repurposing
## An Explainable KG-Based Framework for Bangladeshi Lung Cancer Patients

**Prerequisites:** Notebook 1 must have been run. Upload `nodes.csv` and `edges.csv` from your Kaggle session or attach the processed dataset.

**Task:** Heterogeneous Graph Link Prediction — predicting novel `drug_targets_gene` edges that represent drug repurposing candidates.

---
### Architecture Summary
```
Input KG (nodes.csv + edges.csv)
         │
         ▼
 Node Embedding Layer  (learned, per node type)
         │
         ▼
 R-GCN Encoder         (2-layer Relational Graph Convolution)
         │
         ▼
 Node Embeddings       (32-dim representation per node)
         │
         ▼
 Link Prediction Head  (dot-product decoder + sigmoid)
         │
         ▼
 Drug Repurposing Score (ranked list of drug-gene candidates)
```

In [ ]:
# ==============================================================
# SECTION 0: Install PyTorch Geometric on Kaggle
# ==============================================================
import subprocess, sys

print('Installing PyTorch Geometric...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric'], check=True)

# Optional PyG extensions (faster sparse ops) — install if available
try:
    import torch
    torch_ver = torch.__version__.split('+')[0]
    cuda_ver  = 'cpu'
    pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch_scatter', 'torch_sparse', '-f', pyg_url],
                   capture_output=True)
    print(f'PyG extensions attempted for torch {torch_ver}')
except Exception as e:
    print(f'PyG extensions skipped (non-critical): {e}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'matplotlib', 'seaborn', 'tqdm'], check=True)
print('All dependencies ready.')

In [ ]:
# ==============================================================
# SECTION 0B: Imports & Configuration
# ==============================================================
import os, json, warnings, datetime
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from   tqdm import tqdm

import torch
import torch.nn            as nn
import torch.nn.functional as F
from   torch.optim         import Adam
from   torch.optim.lr_scheduler import ReduceLROnPlateau

import torch_geometric
from torch_geometric.data  import HeteroData
from torch_geometric.nn    import HeteroConv, SAGEConv, Linear, to_hetero
from torch_geometric.utils import negative_sampling, to_undirected
from torch_geometric.transforms import RandomLinkSplit

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────
CONFIG = {
    'embedding_dim'     : 64,
    'hidden_dim'        : 64,
    'output_dim'        : 32,
    'num_layers'        : 2,
    'dropout'           : 0.3,
    'learning_rate'     : 0.005,
    'weight_decay'      : 1e-4,
    'epochs'            : 200,
    'patience'          : 25,
    'batch_size'        : 4096,
    'neg_sampling_ratio': 3,
    'val_ratio'         : 0.10,
    'test_ratio'        : 0.10,
    'seed'              : 42,
    'primary_relation'  : ('Drug', 'drug_targets_gene', 'Gene'),
    'secondary_relation': ('Disease', 'disease_associates_gene', 'Gene'),
    'checkpoint_dir'    : 'results/checkpoints',
    'figures_dir'       : 'results/figures',
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for d in [CONFIG['checkpoint_dir'], CONFIG['figures_dir']]:
    os.makedirs(d, exist_ok=True)

TIMESTAMP = datetime.datetime.now().isoformat()
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'PyG    : {torch_geometric.__version__}')
print(f'Config : embedding_dim={CONFIG["embedding_dim"]}, layers={CONFIG["num_layers"]}, epochs={CONFIG["epochs"]}')

---
## SECTION 1: Load & Inspect the Knowledge Graph

In [ ]:
# ==============================================================
# SECTION 1: Load nodes.csv and edges.csv
# ==============================================================

# Locate files — handles both Kaggle working dir and attached datasets
def find_file(filename, search_roots=None):
    if search_roots is None:
        search_roots = ['/kaggle/working', '/kaggle/input', '.']
    for root in search_roots:
        for dirpath, _, files in os.walk(root):
            if filename in files:
                return os.path.join(dirpath, filename)
    return None

nodes_path = find_file('nodes.csv') or 'data/processed/nodes.csv'
edges_path = find_file('edges.csv') or 'data/processed/edges.csv'

print(f'Loading nodes from: {nodes_path}')
print(f'Loading edges from: {edges_path}')

nodes_df = pd.read_csv(nodes_path, low_memory=False)
edges_df = pd.read_csv(edges_path, low_memory=False)

print(f'\nNodes: {len(nodes_df):,}')
print(nodes_df['node_type'].value_counts().to_string())
print(f'\nEdges: {len(edges_df):,}')
print(edges_df['relation_type'].value_counts().to_string())
print(f'\nEdge weight stats:')
print(edges_df['weight'].describe())

In [ ]:
# ==============================================================
# SECTION 1B: Build Per-Type Node ID Mappings
# ==============================================================
# PyTorch Geometric HeteroData requires per-type 0-indexed node IDs

# Get all node types present in the graph
NODE_TYPES = nodes_df['node_type'].unique().tolist()
RELATION_TYPES = edges_df['relation_type'].unique().tolist()
print(f'Node types  : {NODE_TYPES}')
print(f'Relation types: {RELATION_TYPES}')

# Create per-type local ID mapping
# global node_id  ->  (node_type, local_type_id)
type_to_nodes  = {}  # node_type -> DataFrame
type_local_map = {}  # global node_id -> local type-specific id

for ntype in NODE_TYPES:
    subset = nodes_df[nodes_df['node_type'] == ntype].reset_index(drop=True)
    type_to_nodes[ntype] = subset
    for local_id, row in subset.iterrows():
        type_local_map[int(row['node_id'])] = (ntype, local_id)

# Build per-type count dict
num_nodes_per_type = {ntype: len(df) for ntype, df in type_to_nodes.items()}
print('\nPer-type node counts:')
for k, v in num_nodes_per_type.items():
    print(f'  {k:<25}: {v:>7,}')

---
## SECTION 2: Build PyTorch Geometric HeteroData Object

In [ ]:
# ==============================================================
# SECTION 2: Build HeteroData graph object
# ==============================================================
# Node features: we use learned embeddings (no raw features)
# Each node type gets an integer index — the embedding layer maps it

data = HeteroData()

# Assign placeholder features (node indices) for each type
for ntype, count in num_nodes_per_type.items():
    # x will be used as an embedding index (0 to count-1)
    data[ntype].x        = torch.arange(count, dtype=torch.long)
    data[ntype].num_nodes = count

# Build edge index tensors for each relation type
# Edge triplets: (src_type, relation, dst_type)

def get_node_type(global_id):
    return type_local_map.get(int(global_id), (None, None))

# Group edges by (src_type, relation_type, dst_type)
print('Building heterogeneous edge index tensors...')
edge_groups = {}

for _, row in tqdm(edges_df.iterrows(), total=len(edges_df), desc='Edges'):
    src_type, src_local = get_node_type(row['src_node_id'])
    dst_type, dst_local = get_node_type(row['dst_node_id'])

    if src_type is None or dst_type is None:
        continue

    rel   = str(row['relation_type'])
    key   = (src_type, rel, dst_type)
    wt    = float(row['weight']) if not pd.isna(row['weight']) else 1.0

    if key not in edge_groups:
        edge_groups[key] = {'src': [], 'dst': [], 'weight': []}

    edge_groups[key]['src'].append(src_local)
    edge_groups[key]['dst'].append(dst_local)
    edge_groups[key]['weight'].append(wt)

# Assign to HeteroData
print(f'\nAssigning {len(edge_groups)} relation types to HeteroData...')
for (src_type, rel, dst_type), edges in edge_groups.items():
    src_t = torch.tensor(edges['src'], dtype=torch.long)
    dst_t = torch.tensor(edges['dst'], dtype=torch.long)
    wt_t  = torch.tensor(edges['weight'], dtype=torch.float)

    data[src_type, rel, dst_type].edge_index = torch.stack([src_t, dst_t])
    data[src_type, rel, dst_type].edge_attr  = wt_t
    print(f'  ({src_type}, {rel}, {dst_type}): {src_t.shape[0]:>7,} edges')

print(f'\nHeteroData object built successfully.')
print(f'  Node types: {data.node_types}')
print(f'  Edge types: {len(data.edge_types)}')

In [ ]:
# ==============================================================
# SECTION 2B: Train / Val / Test Split on Primary Relation
# ==============================================================
# Primary link prediction task: Drug --drug_targets_gene--> Gene

PRIMARY_SRC, PRIMARY_REL, PRIMARY_DST = CONFIG['primary_relation']
print(f'Primary prediction task: ({PRIMARY_SRC}, {PRIMARY_REL}, {PRIMARY_DST})')

primary_key = (PRIMARY_SRC, PRIMARY_REL, PRIMARY_DST)

if primary_key not in [tuple(et) for et in data.edge_types]:
    # Try to find the closest matching edge type
    print(f'Primary relation not found in HeteroData. Available:')
    for et in data.edge_types:
        print(f'  {et}')
    raise ValueError('Primary relation missing — check Notebook 1 output')

primary_edges = data[PRIMARY_SRC, PRIMARY_REL, PRIMARY_DST].edge_index
n_edges       = primary_edges.shape[1]
print(f'Total primary edges: {n_edges}')

# Shuffle and split
perm       = torch.randperm(n_edges, generator=torch.Generator().manual_seed(CONFIG['seed']))
n_val      = int(n_edges * CONFIG['val_ratio'])
n_test     = int(n_edges * CONFIG['test_ratio'])
n_train    = n_edges - n_val - n_test

train_idx  = perm[:n_train]
val_idx    = perm[n_train:n_train + n_val]
test_idx   = perm[n_train + n_val:]

train_edges = primary_edges[:, train_idx]
val_edges   = primary_edges[:, val_idx]
test_edges  = primary_edges[:, test_idx]

print(f'Train edges: {train_edges.shape[1]}')
print(f'Val   edges: {val_edges.shape[1]}')
print(f'Test  edges: {test_edges.shape[1]}')

# For training, use the full graph (all relation types) as message-passing graph
# but only train/val/test on the primary edges
train_data = data.clone()
train_data[PRIMARY_SRC, PRIMARY_REL, PRIMARY_DST].edge_index = train_edges

---
## SECTION 3: R-GCN Model Architecture
```
Input:  Node type embeddings  (dim = embedding_dim)
  │
  ├── Layer 1: HeteroConv (SAGEConv per relation)
  │           hidden_dim = 64, activation = ReLU, dropout = 0.3
  │
  └── Layer 2: HeteroConv (SAGEConv per relation)
              output_dim = 32
  │
Link Prediction: dot(src_emb, dst_emb) → sigmoid → P(edge exists)
```

In [ ]:
# ==============================================================
# SECTION 3: R-GCN Model Definition
# ==============================================================

class NodeEmbeddingLayer(nn.Module):
    """Learned embedding table for each node type (no raw features)."""
    def __init__(self, num_nodes_dict, embedding_dim):
        super().__init__()
        self.embeddings = nn.ModuleDict({
            ntype: nn.Embedding(n, embedding_dim)
            for ntype, n in num_nodes_dict.items()
        })
        # Xavier initialization
        for emb in self.embeddings.values():
            nn.init.xavier_uniform_(emb.weight)

    def forward(self, x_dict):
        return {ntype: self.embeddings[ntype](x)
                for ntype, x in x_dict.items()}


class RelationalGCN(nn.Module):
    """R-GCN Encoder using HeteroConv with SAGEConv per relation."""
    def __init__(self, hidden_dim, output_dim, metadata, num_layers=2, dropout=0.3):
        super().__init__()
        self.dropout   = dropout
        self.num_layers = num_layers
        self.convs = nn.ModuleList()

        for layer_idx in range(num_layers):
            out_dim = output_dim if layer_idx == num_layers - 1 else hidden_dim
            conv = HeteroConv({
                edge_type: SAGEConv((-1, -1), out_dim, add_self_loops=False)
                for edge_type in metadata[1]
            }, aggr='mean')  # Mean aggregation across relation types
            self.convs.append(conv)

    def forward(self, x_dict, edge_index_dict):
        for i, conv in enumerate(self.convs):
            x_dict = conv(x_dict, edge_index_dict)
            if i < self.num_layers - 1:
                x_dict = {k: F.relu(v) for k, v in x_dict.items()}
                x_dict = {k: F.dropout(v, p=self.dropout, training=self.training)
                          for k, v in x_dict.items()}
        return x_dict


class LinkPredictor(nn.Module):
    """Dot-product link prediction decoder with optional MLP."""
    def __init__(self, embedding_dim):
        super().__init__()
        # Bilinear scoring: score = src^T W dst
        self.W = nn.Linear(embedding_dim, embedding_dim, bias=False)

    def forward(self, src_emb, dst_emb):
        # Score: (N,) — higher = more likely to be linked
        return (self.W(src_emb) * dst_emb).sum(dim=-1)


class DrugRepurposingRGCN(nn.Module):
    """Full R-GCN model for heterogeneous drug repurposing KG."""
    def __init__(self, num_nodes_dict, embedding_dim, hidden_dim,
                 output_dim, metadata, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding_layer = NodeEmbeddingLayer(num_nodes_dict, embedding_dim)
        self.encoder         = RelationalGCN(hidden_dim, output_dim,
                                             metadata, num_layers, dropout)
        self.link_predictor  = LinkPredictor(output_dim)

    def encode(self, x_dict, edge_index_dict):
        h = self.embedding_layer(x_dict)
        h = self.encoder(h, edge_index_dict)
        return h

    def decode(self, embeddings, src_type, dst_type, edge_index):
        src_emb = embeddings[src_type][edge_index[0]]
        dst_emb = embeddings[dst_type][edge_index[1]]
        return self.link_predictor(src_emb, dst_emb)

    def forward(self, x_dict, edge_index_dict, pos_edge_index, neg_edge_index,
                src_type, dst_type):
        h = self.encode(x_dict, edge_index_dict)
        pos_scores = self.decode(h, src_type, dst_type, pos_edge_index)
        neg_scores = self.decode(h, src_type, dst_type, neg_edge_index)
        return pos_scores, neg_scores


# Instantiate the model
model = DrugRepurposingRGCN(
    num_nodes_dict = num_nodes_per_type,
    embedding_dim  = CONFIG['embedding_dim'],
    hidden_dim     = CONFIG['hidden_dim'],
    output_dim     = CONFIG['output_dim'],
    metadata       = (data.node_types, data.edge_types),
    num_layers     = CONFIG['num_layers'],
    dropout        = CONFIG['dropout'],
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model instantiated.')
print(f'Total trainable parameters: {total_params:,}')
print(model)

---
## SECTION 4: Training Loop with Negative Sampling

In [ ]:
# ==============================================================
# SECTION 4A: Helper Functions
# ==============================================================

def generate_negative_samples(pos_edge_index, num_src_nodes, num_dst_nodes, ratio=3):
    """Generate negative edges via random sampling."""
    num_neg = pos_edge_index.shape[1] * ratio
    neg_src = torch.randint(0, num_src_nodes, (num_neg,))
    neg_dst = torch.randint(0, num_dst_nodes, (num_neg,))
    return torch.stack([neg_src, neg_dst])


def compute_loss(pos_scores, neg_scores):
    """Binary cross-entropy loss with label smoothing."""
    pos_labels = torch.ones(pos_scores.shape[0],  device=DEVICE) * 0.95  # Label smoothing
    neg_labels = torch.zeros(neg_scores.shape[0], device=DEVICE) * 0.05
    scores = torch.cat([pos_scores, neg_scores])
    labels = torch.cat([pos_labels, neg_labels])
    return F.binary_cross_entropy_with_logits(scores, labels)


@torch.no_grad()
def evaluate(model, data, eval_edges, src_type, dst_type,
             num_src, num_dst, neg_ratio=10):
    """Compute AUROC and AUPRC on eval edges."""
    model.eval()
    x_dict         = {k: v.to(DEVICE) for k, v in data.x_dict.items()}
    edge_index_dict = {k: v.to(DEVICE) for k, v in data.edge_index_dict.items()}

    h = model.encode(x_dict, edge_index_dict)

    # Positive scores
    pos_ei  = eval_edges.to(DEVICE)
    pos_sc  = torch.sigmoid(model.decode(h, src_type, dst_type, pos_ei)).cpu().numpy()

    # Negative scores (sample more for better AUROC estimate)
    neg_ei  = generate_negative_samples(eval_edges, num_src, num_dst, ratio=neg_ratio).to(DEVICE)
    neg_sc  = torch.sigmoid(model.decode(h, src_type, dst_type, neg_ei)).cpu().numpy()

    scores  = np.concatenate([pos_sc, neg_sc])
    labels  = np.concatenate([np.ones(len(pos_sc)), np.zeros(len(neg_sc))])

    auroc   = roc_auc_score(labels, scores) if len(np.unique(labels)) > 1 else 0.5
    auprc   = average_precision_score(labels, scores)

    # Hits@K
    all_scores = np.concatenate([pos_sc, neg_sc])
    threshold  = np.percentile(all_scores, 90)  # Top 10%
    hits_at_10 = np.mean(pos_sc >= threshold)

    return {'auroc': auroc, 'auprc': auprc, 'hits@10': hits_at_10}

In [ ]:
# ==============================================================
# SECTION 4B: Training Loop
# ==============================================================

optimizer = Adam(model.parameters(),
                 lr=CONFIG['learning_rate'],
                 weight_decay=CONFIG['weight_decay'])
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5,
                               patience=10, verbose=True)

NUM_SRC  = num_nodes_per_type[PRIMARY_SRC]
NUM_DST  = num_nodes_per_type[PRIMARY_DST]

# Move graph data to device
x_dict_dev         = {k: v.to(DEVICE) for k, v in train_data.x_dict.items()}
edge_index_dict_dev = {k: v.to(DEVICE) for k, v in train_data.edge_index_dict.items()}
train_edges_dev     = train_edges.to(DEVICE)
val_edges_dev       = val_edges.to(DEVICE)

history = {
    'train_loss': [], 'val_auroc': [], 'val_auprc': [], 'val_hits': []
}
best_auroc  = 0.0
patience_ct = 0
best_epoch  = 0

print('Starting R-GCN Training...')
print(f'{'Epoch':>6} | {'Train Loss':>12} | {'Val AUROC':>10} | {'Val AUPRC':>10} | {'Hits@10':>8}')
print('-' * 60)

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()

    # Generate fresh negative samples each epoch
    neg_ei = generate_negative_samples(
        train_edges_dev, NUM_SRC, NUM_DST,
        ratio=CONFIG['neg_sampling_ratio']
    ).to(DEVICE)

    optimizer.zero_grad()
    pos_scores, neg_scores = model(
        x_dict_dev, edge_index_dict_dev,
        train_edges_dev, neg_ei,
        PRIMARY_SRC, PRIMARY_DST
    )
    loss = compute_loss(pos_scores, neg_scores)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    history['train_loss'].append(loss.item())

    # Evaluate every 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        metrics = evaluate(
            model, train_data, val_edges,
            PRIMARY_SRC, PRIMARY_DST,
            NUM_SRC, NUM_DST
        )
        val_auroc = metrics['auroc']
        val_auprc = metrics['auprc']
        hits      = metrics['hits@10']

        history['val_auroc'].append(val_auroc)
        history['val_auprc'].append(val_auprc)
        history['val_hits'].append(hits)

        scheduler.step(val_auroc)

        print(f'{epoch:>6} | {loss.item():>12.4f} | {val_auroc:>10.4f} | {val_auprc:>10.4f} | {hits:>8.4f}')

        # Save best model
        if val_auroc > best_auroc:
            best_auroc  = val_auroc
            best_epoch  = epoch
            patience_ct = 0
            torch.save({
                'epoch'       : epoch,
                'model_state' : model.state_dict(),
                'optimizer'   : optimizer.state_dict(),
                'val_auroc'   : val_auroc,
                'val_auprc'   : val_auprc,
                'config'      : CONFIG,
            }, f'{CONFIG["checkpoint_dir"]}/best_model.pt')
        else:
            patience_ct += 1

        if patience_ct >= CONFIG['patience'] // 10:
            print(f'\nEarly stopping at epoch {epoch} (best AUROC: {best_auroc:.4f} at epoch {best_epoch})')
            break

print(f'\nTraining complete.')
print(f'Best validation AUROC: {best_auroc:.4f} (epoch {best_epoch})')

---
## SECTION 5: Final Evaluation on Test Set

In [ ]:
# ==============================================================
# SECTION 5: Load Best Model & Final Test Evaluation
# ==============================================================

checkpoint = torch.load(f'{CONFIG["checkpoint_dir"]}/best_model.pt', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
print(f'Loaded best model from epoch {checkpoint["epoch"]} (val AUROC={checkpoint["val_auroc"]:.4f})')

# Full test evaluation
test_metrics = evaluate(
    model, train_data, test_edges,
    PRIMARY_SRC, PRIMARY_DST,
    NUM_SRC, NUM_DST, neg_ratio=10
)

print('\n' + '='*50)
print('FINAL TEST SET PERFORMANCE')
print('='*50)
print(f'  AUROC   : {test_metrics["auroc"]:.4f}')
print(f'  AUPRC   : {test_metrics["auprc"]:.4f}')
print(f'  Hits@10 : {test_metrics["hits@10"]:.4f}')

# Save results
results_dict = {
    'test_auroc'    : test_metrics['auroc'],
    'test_auprc'    : test_metrics['auprc'],
    'test_hits@10'  : test_metrics['hits@10'],
    'val_auroc'     : checkpoint['val_auroc'],
    'val_auprc'     : checkpoint['val_auprc'],
    'best_epoch'    : checkpoint['epoch'],
    'total_nodes'   : len(nodes_df),
    'total_edges'   : len(edges_df),
    'timestamp'     : TIMESTAMP,
    'config'        : CONFIG,
}
with open(f'{CONFIG["checkpoint_dir"]}/test_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2, default=str)
print(f'\nResults saved to {CONFIG["checkpoint_dir"]}/test_results.json')

In [ ]:
# ==============================================================
# SECTION 5B: Training Curves
# ==============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('R-GCN Training History', fontsize=14, fontweight='bold')

axes[0].plot(history['train_loss'], color='#2563eb', linewidth=1.5)
axes[0].set_title('Training Loss (BCE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

eval_epochs = list(range(10, len(history['val_auroc']) * 10 + 1, 10))
axes[1].plot(eval_epochs, history['val_auroc'], color='#16a34a', linewidth=1.5, label='AUROC')
axes[1].plot(eval_epochs, history['val_auprc'], color='#dc2626', linewidth=1.5, linestyle='--', label='AUPRC')
axes[1].axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Random baseline')
axes[1].set_title('Validation AUROC & AUPRC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(eval_epochs, history['val_hits'], color='#7c3aed', linewidth=1.5)
axes[2].set_title('Validation Hits@10')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Hits@10')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CONFIG["figures_dir"]}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

---
## SECTION 6: Drug Repurposing Candidate Generation
Score ALL drug-gene pairs not in the training set and rank them as repurposing candidates.

In [ ]:
# ==============================================================
# SECTION 6: Generate Drug Repurposing Candidates
# ==============================================================

print('Generating drug repurposing candidate scores...')
model.eval()

with torch.no_grad():
    x_dict_all = {k: v.to(DEVICE) for k, v in train_data.x_dict.items()}
    ei_dict_all = {k: v.to(DEVICE) for k, v in train_data.edge_index_dict.items()}
    embeddings  = model.encode(x_dict_all, ei_dict_all)

drug_emb = embeddings[PRIMARY_SRC].cpu()  # (num_drugs, output_dim)
gene_emb = embeddings[PRIMARY_DST].cpu()  # (num_genes, output_dim)

print(f'Drug embeddings: {drug_emb.shape}')
print(f'Gene embeddings: {gene_emb.shape}')

# Score all drug-gene pairs
# Score matrix: (num_drugs, num_genes) — dot product
score_matrix = torch.sigmoid(
    torch.mm(model.link_predictor.W(drug_emb), gene_emb.T)
).numpy()

# Build known edge set (train + val + test) to exclude
known_edges = set(zip(
    primary_edges[0].numpy().tolist(),
    primary_edges[1].numpy().tolist()
))

# Build drug and gene metadata
drug_meta = type_to_nodes[PRIMARY_SRC][['node_id','label','canonical_id']].copy()
drug_meta['local_id'] = range(len(drug_meta))
drug_meta = drug_meta.reset_index(drop=True)

gene_meta = type_to_nodes[PRIMARY_DST][['node_id','label','canonical_id']].copy()
gene_meta['local_id'] = range(len(gene_meta))
gene_meta = gene_meta.reset_index(drop=True)

# Collect top candidates
candidates = []
for d_idx in range(len(drug_meta)):
    drug_scores = score_matrix[d_idx]
    # Top 20 gene predictions for this drug
    top_gene_ids = np.argsort(drug_scores)[::-1][:20]
    for g_idx in top_gene_ids:
        if (d_idx, int(g_idx)) in known_edges:
            continue  # Skip known edges — only novel predictions
        candidates.append({
            'drug_local_id'    : d_idx,
            'gene_local_id'    : int(g_idx),
            'drug_name'        : drug_meta.loc[d_idx, 'label'],
            'drug_canonical_id': drug_meta.loc[d_idx, 'canonical_id'],
            'gene_symbol'      : gene_meta.loc[int(g_idx), 'label'],
            'gene_canonical_id': gene_meta.loc[int(g_idx), 'canonical_id'],
            'repurposing_score': float(drug_scores[g_idx]),
            'is_novel'         : True,
        })

candidates_df = pd.DataFrame(candidates).sort_values(
    'repurposing_score', ascending=False
)
candidates_df.to_csv('results/drug_repurposing_candidates.csv', index=False)

print(f'\nGenerated {len(candidates_df)} novel drug-gene repurposing candidates')
print('\nTop 20 Repurposing Candidates:')
print(candidates_df[['drug_name','gene_symbol','repurposing_score']].head(20).to_string(index=False))

In [ ]:
# ==============================================================
# SECTION 6B: Save Node Embeddings for XAI (Notebook 3)
# ==============================================================

print('Saving node embeddings for XAI path extraction...')

with torch.no_grad():
    all_embs = model.encode(
        {k: v.to(DEVICE) for k, v in train_data.x_dict.items()},
        {k: v.to(DEVICE) for k, v in train_data.edge_index_dict.items()}
    )

emb_records = []
for ntype, emb_tensor in all_embs.items():
    emb_np   = emb_tensor.cpu().numpy()
    meta_df  = type_to_nodes[ntype].reset_index(drop=True)
    emb_cols = {f'emb_{i}': emb_np[:, i] for i in range(emb_np.shape[1])}
    node_df  = meta_df[['node_id','canonical_id','label']].copy()
    node_df['node_type'] = ntype
    for col, vals in emb_cols.items():
        node_df[col] = vals
    emb_records.append(node_df)

all_embeddings_df = pd.concat(emb_records, ignore_index=True)
all_embeddings_df.to_csv('results/node_embeddings.csv', index=False)
print(f'Saved embeddings for {len(all_embeddings_df):,} nodes → results/node_embeddings.csv')
print(f'Embedding dimension: {CONFIG["output_dim"]}')
print('Ready for Notebook 3: XAI Path Extraction')

---
## SECTION 7: Model Summary & Personalization Hook Documentation

In [ ]:
# ==============================================================
# SECTION 7: Personalization Hook (VAF Integration Point)
# ==============================================================
# This section documents WHERE Bangladeshi patient VAF data
# integrates into the trained model for personalization.

print('='*65)
print('PERSONALIZATION HOOK: Bangladeshi Patient VAF Integration')
print('='*65)
print('''
When primary Bangladeshi patient data is available (post-IRB clearance),
personalization occurs in TWO steps:

STEP A — Edge Weight Modification (before inference):
  For each patient:
    1. Load patient somatic mutation profile (MAF file, VAF scores)
    2. For each mutated gene G with VAF v:
       - Find all disease_associates_gene edges: (Disease → G)
       - Multiply edge weight by (1 + vaf_score)
       - This up-weights genes with higher mutation burden
    3. Re-encode the MODIFIED graph using the trained R-GCN

STEP B — Personalized Scoring (during inference):
  personalized_score(drug, gene, patient) =
    global_score(drug, gene)          # from trained model
    × vaf_weight(gene, patient)       # from patient mutation
    × expression_weight(gene)         # from GTEx normal lung

CODE TEMPLATE:
''')

personalization_code = '''
def personalize_for_patient(model, base_graph, patient_vaf_dict, embeddings):
    """
    patient_vaf_dict: {gene_symbol: vaf_score}  e.g. {'EGFR': 0.42, 'TP53': 0.61}
    """
    import copy
    personalized = copy.deepcopy(base_graph)

    # Re-weight disease-gene edges by VAF
    disease_gene_key = ('Disease', 'disease_associates_gene', 'Gene')
    if disease_gene_key in personalized.edge_types:
        ei  = personalized[disease_gene_key].edge_index
        wts = personalized[disease_gene_key].edge_attr.clone()
        for gene_sym, vaf in patient_vaf_dict.items():
            gene_local_id = gene_meta[gene_meta['label'] == gene_sym]['local_id']
            if len(gene_local_id) > 0:
                mask = (ei[1] == gene_local_id.values[0])
                wts[mask] *= (1.0 + float(vaf))  # VAF up-weighting
        wts = wts.clamp(0.0, 1.0)  # Normalise back to [0,1]
        personalized[disease_gene_key].edge_attr = wts

    # Re-encode with personalized graph
    model.eval()
    with torch.no_grad():
        personalized_embs = model.encode(
            {k: v.to(DEVICE) for k, v in personalized.x_dict.items()},
            {k: v.to(DEVICE) for k, v in personalized.edge_index_dict.items()}
        )
    return personalized_embs
'''
print(personalization_code)

with open('results/personalization_hook.py', 'w') as f:
    f.write(personalization_code)
print('Personalization hook saved to results/personalization_hook.py')

In [ ]:
# ==============================================================
# SECTION 7B: Final Summary
# ==============================================================

print('='*65)
print('NOTEBOOK 2 COMPLETE — R-GCN MODEL TRAINING')
print('='*65)

print(f'''
Model Performance:
  Test AUROC   : {test_metrics["auroc"]:.4f}
  Test AUPRC   : {test_metrics["auprc"]:.4f}
  Test Hits@10 : {test_metrics["hits@10"]:.4f}

Output Files:
  results/checkpoints/best_model.pt          — Trained R-GCN weights
  results/checkpoints/test_results.json      — Performance metrics
  results/drug_repurposing_candidates.csv    — Ranked novel drug-gene pairs
  results/node_embeddings.csv                — All node embeddings (32-dim)
  results/figures/training_curves.png        — Training history plots
  results/personalization_hook.py            — VAF personalization template

Next Steps:
  Notebook 3: XAI Path Extraction
    - Load node_embeddings.csv
    - Implement graph traversal to extract mechanistic paths
    - Validate paths against DrugMechDB reference paths
    - Generate human-readable explanations per candidate drug
''')

print(f'Timestamp: {TIMESTAMP}')